## 1. PEAS Framework & Environment Characteristics
#### A. PEAS Framework for the Self-Driving Trailer Agent
**Performance Measure:**
Find a successfull route to reach the destination (Kisumu from Nairobi).
Use minimal computation time to find the route.
Minimize total distance travelled along said route.
Trip duration and whether delivery deadlines are met.
Safety (no collisions), rule compliance (obeying speed limits, lane keeping), and passenger/cargo comfort (smooth braking and maintaining stable movement during transportation.).

**Environment:**
The physical highway network in Kenya (specifically Nairobi, Central Highlands, Rift Valley, and Lake Victoria Basin) including the A104, B1, B3, and A1 corridors. 
Different road conditions, road construction areas, speed bumps, potholes, and changes in terrain such as the Rift Valley escarpments. Other road users, including heavy trucks, matatus, passenger vehicles, motorcycles (boda-bodas), pedestrians, and wandering livestock. Weather conditions such as heavy rain and fog, especially around areas such as Limuru and Mau Summit.
Google Maps data representation of these regions with locations(nodes), roads(edges) and distances(weights/costs).

**Actuators:**
The mechanism to output the calculated optimal path like a GUI display showing the route, a printed list of waypoints, or map rendering.
Hardware components: Electronic throttle/accelerator control. Air-braking system and retarder. Steering mechanisms for controlling the steering angle. Transmission and gear selection. Signalling and lighting systems, including turn indicators, headlights, brake lights, and hazard lights.

**Sensors:**
Google Maps Search/Directions API used to retrieve nodes, edges, and distance inputs.
User interface to receive input about starting point and destination.
Real-time monitoring through cameras (for lane detection, traffic lights, obstacles), long-range LiDAR and radar for detecting obstacles and supporting adaptive cruise control. Cameras for detecting lanes, road signs, traffic lights, and pedestrians. GPS/GNSS and an IMU (Inertial Measurement Unit) for determining the vehicle’s position and orientation. Wheel-speed sensors and engine diagnostics for monitoring vehicle speed, fuel level, tyre pressure, and other vehicle conditions.

#### B. Environment Dimensions
1. **Partially Observable vs. Fully Observable:** **Partially Observable.** The trailer agent cannot know everything happening on the road at a given time. It may not know about traffic jams ahead, broken-down vehicles or sudden changes in weather around areas like Mau Summit.

2. **Deterministic vs. Stochastic:** **Stochastic.** Road and traffic conditions can change unexpectedly. Other drivers may behave unpredictably, pedestrians may cross the road suddenly and weather conditions may change, making it difficult to predict the exact outcome of an action.

3. **Episodic vs. Sequential:** **Sequential.** Each decision affects what happens next. For example, choosing a route through the A104 instead of the B3 via Narok affects the distance travelled, fuel consumption and the route options available later.

4. **Static vs. Dynamic:** **Dynamic.** The road environment changes while the agent is making decisions. Traffic levels, the positions of other vehicles and road conditions can change at any time.

5. **Discrete vs. Continuous:** **Continuous.** The trailer's speed, position, steering angle and distance travelled can take different values within a range. However, when planning routes, road junctions can be represented as discrete points on a map.

## 2. Directed Graph & Search Tree (with Abstraction)
#### A. Directed Graph of Candidate Road Networks

### Applying Abstraction (removing unnecessary details)
* We ignore:
  *the physical curvature of the roads, speed limits, traffic lights, and scenery.
  *minor villages or sub-locations.

* We model the environment as a directed graph using:
  * **Nodes (Vertices):** Major towns (Nairobi, Naivasha, Nakuru, Kericho, Eldoret, Kisumu, Narok, Bomet, etc.).
  * **Edges (Arcs):** The major highways connecting these towns.

### Developing the Directed Graph

Using the Google Maps view of the specified Kenyan regions, we extracted a simplified graph with these connections:

```mermaid
graph TD
    Nairobi-->|22.3km|Gitaru
    Nairobi-->|15.5km|Ruaka
    Nairobi-->|15.5km|Kiambu
    Nairobi-->|24.6|Ruiru
    Nairobi-->|53.8km|Maai-Mahiu
    Gitaru-->|20.1km|Limuru
    Ruaka-->|21km|Limuru
    Kiambu-->|25.7km|Limuru
    Limuru-->|56.1km|Naivasha
    Naivasha-->|30.4km|Gilgil
    Gilgil-->|42.6km|Nakuru
    Nakuru-->|94.5km|Timboroa
    Nakuru-->|63.1km|Eldama-Ravine
    Timboroa-->|61.7km|Eldoret
    Eldama-Ravine-->|108km|Eldoret
    Eldoret-->|46km|Kapsabet
    Eldoret-->|106km|Kakamega
    Kapsabet-->|80.3km|Kisumu
    Kakamega-->|53.4km|Kisumu
    Ruiru-->|21.4km|Thika
    Thika-->|63.9km|Sagana
    Thika-->|86.8km|Othaya
    Sagana-->|156km|Nyahururu
    Othaya-->|16.9km|Nyeri
    Nyeri-->|104km|Nyahururu
    Nyahururu-->|63.3km|Nakuru
    Nakuru-->|107km|Kericho
    Kericho-->|60.9km|Ahero
    Maai-Mahiu-->|89.1km|Narok
    Narok-->|81km|Bomet
    Bomet-->|82.4km|Kisii
    Kisii-->|52km|Homa-Bay
    Homa-Bay-->|84.5km|Ahero
    Kisii-->|77.4km|Awach
    Awach-->|14.1km|Ahero
    Ahero-->|23.5km|Kisumu

```


#### B. Search Tree (Applying Abstraction)
```mermaid
graph TD
    N[Nairobi] -->|22.3km| Git[Gitaru]
    N -->|15.5km| Rua[Ruaka]
    N -->|15.5km| Kia[Kiambu]
    N -->|24.6km| Rui[Ruiru]
    N -->|53.8km| MM[Maai-Mahiu]

    %% Branch 1: Gitaru
    Git -->|20.1km| Lim1[Limuru]
    Lim1 -->|56.1km| Nai1[Naivasha]
    Nai1 -->|30.4km| Gil1[Gilgil]
    Gil1 -->|42.6km| Nak1[Nakuru]
    Nak1 -->|94.5km| Tim1[Timboroa]
    Tim1 -->|61.7km| Eld1A[Eldoret]
    Eld1A -->|46km| Kap1A[Kapsabet]
    Kap1A -->|80.3km| Kisumu1A[Kisumu]
    Eld1A -->|106km| Kak1A[Kakamega]
    Kak1A -->|53.4km| Kisumu1B[Kisumu]
    Nak1 -->|63.1km| ER1[Eldama-Ravine]
    ER1 -->|108km| Eld1B[Eldoret]
    Eld1B -->|46km| Kap1B[Kapsabet]
    Kap1B -->|80.3km| Kisumu1C[Kisumu]
    Eld1B -->|106km| Kak1B[Kakamega]
    Kak1B -->|53.4km| Kisumu1D[Kisumu]
    Nak1 -->|107km| Ker1[Kericho]
    Ker1 -->|60.9km| Ahe1[Ahero]
    Ahe1 -->|23.5km| Kisumu1E[Kisumu]

    %% Branch 2: Ruaka
    Rua -->|21km| Lim2[Limuru]
    Lim2 -->|56.1km| Nai2[Naivasha]
    Nai2 -->|30.4km| Gil2[Gilgil]
    Gil2 -->|42.6km| Nak2[Nakuru]
    Nak2 -->|94.5km| Tim2[Timboroa]
    Tim2 -->|61.7km| Eld2A[Eldoret]
    Eld2A -->|46km| Kap2A[Kapsabet]
    Kap2A -->|80.3km| Kisumu2A[Kisumu]
    Eld2A -->|106km| Kak2A[Kakamega]
    Kak2A -->|53.4km| Kisumu2B[Kisumu]
    Nak2 -->|63.1km| ER2[Eldama-Ravine]
    ER2 -->|108km| Eld2B[Eldoret]
    Eld2B -->|46km| Kap2B[Kapsabet]
    Kap2B -->|80.3km| Kisumu2C[Kisumu]
    Eld2B -->|106km| Kak2B[Kakamega]
    Kak2B -->|53.4km| Kisumu2D[Kisumu]
    Nak2 -->|107km| Ker2[Kericho]
    Ker2 -->|60.9km| Ahe2[Ahero]
    Ahe2 -->|23.5km| Kisumu2E[Kisumu]

    %% Branch 3: Kiambu
    Kia -->|25.7km| Lim3[Limuru]
    Lim3 -->|56.1km| Nai3[Naivasha]
    Nai3 -->|30.4km| Gil3[Gilgil]
    Gil3 -->|42.6km| Nak3[Nakuru]
    Nak3 -->|94.5km| Tim3[Timboroa]
    Tim3 -->|61.7km| Eld3A[Eldoret]
    Eld3A -->|46km| Kap3A[Kapsabet]
    Kap3A -->|80.3km| Kisumu3A[Kisumu]
    Eld3A -->|106km| Kak3A[Kakamega]
    Kak3A -->|53.4km| Kisumu3B[Kisumu]
    Nak3 -->|63.1km| ER3[Eldama-Ravine]
    ER3 -->|108km| Eld3B[Eldoret]
    Eld3B -->|46km| Kap3B[Kapsabet]
    Kap3B -->|80.3km| Kisumu3C[Kisumu]
    Eld3B -->|106km| Kak3B[Kakamega]
    Kak3B -->|53.4km| Kisumu3D[Kisumu]
    Nak3 -->|107km| Ker3[Kericho]
    Ker3 -->|60.9km| Ahe3[Ahero]
    Ahe3 -->|23.5km| Kisumu3E[Kisumu]

    %% Branch 4: Ruiru (Thika)
    Rui -->|21.4km| Thi[Thika]
    Thi -->|63.9km| Sag[Sagana]
    Sag -->|156km| Nya1[Nyahururu]
    Nya1 -->|63.3km| Nak4[Nakuru]
    Nak4 -->|94.5km| Tim4[Timboroa]
    Tim4 -->|61.7km| Eld4A[Eldoret]
    Eld4A -->|46km| Kap4A[Kapsabet]
    Kap4A -->|80.3km| Kisumu4A[Kisumu]
    Eld4A -->|106km| Kak4A[Kakamega]
    Kak4A -->|53.4km| Kisumu4B[Kisumu]
    Nak4 -->|63.1km| ER4[Eldama-Ravine]
    ER4 -->|108km| Eld4B[Eldoret]
    Eld4B -->|46km| Kap4B[Kapsabet]
    Kap4B -->|80.3km| Kisumu4C[Kisumu]
    Eld4B -->|106km| Kak4B[Kakamega]
    Kak4B -->|53.4km| Kisumu4D[Kisumu]
    Nak4 -->|107km| Ker4[Kericho]
    Ker4 -->|60.9km| Ahe4[Ahero]
    Ahe4 -->|23.5km| Kisumu4E[Kisumu]

    %% Branch 4b: Ruiru (Othaya)
    Thi -->|86.8km| Oth[Othaya]
    Oth -->|16.9km| Nye[Nyeri]
    Nye -->|104km| Nya2[Nyahururu]
    Nya2 -->|63.3km| Nak5[Nakuru]
    Nak5 -->|94.5km| Tim5[Timboroa]
    Tim5 -->|61.7km| Eld5A[Eldoret]
    Eld5A -->|46km| Kap5A[Kapsabet]
    Kap5A -->|80.3km| Kisumu5A[Kisumu]
    Eld5A -->|106km| Kak5A[Kakamega]
    Kak5A -->|53.4km| Kisumu5B[Kisumu]
    Nak5 -->|63.1km| ER5[Eldama-Ravine]
    ER5 -->|108km| Eld5B[Eldoret]
    Eld5B -->|46km| Kap5B[Kapsabet]
    Kap5B -->|80.3km| Kisumu5C[Kisumu]
    Eld5B -->|106km| Kak5B[Kakamega]
    Kak5B -->|53.4km| Kisumu5D[Kisumu]
    Nak5 -->|107km| Ker5[Kericho]
    Ker5 -->|60.9km| Ahe5[Ahero]
    Ahe5 -->|23.5km| Kisumu5E[Kisumu]

    %% Branch 5: Maai-Mahiu
    MM -->|89.1km| Nar[Narok]
    Nar -->|81km| Bom[Bomet]
    Bom -->|82.4km| Kisii[Kisii]
    Kisii -->|52km| HB[Homa-Bay]
    HB -->|84.5km| Ahe6[Ahero]
    Ahe6 -->|23.5km| Kisumu6A[Kisumu]
    Kisii -->|77.4km| Awa[Awach]
    Awa -->|14.1km| Ahe7[Ahero]
    Ahe7 -->|23.5km| Kisumu6B[Kisumu]
```

## 3. Path Cost Function & Optimal Path Determination
#### A. Path Cost Evaluation
**Cost (**$g(n)$**):** The physical distance in kilometres between towns.

* Our team used the Google Maps search feature to find the exact driving distance for each edge in the graph as indicated by the edge labels displaying distance in kilometres between towns.

#### B. Optimal Path Conclusion
Since we are looking for the *optimal* path based on distance along roads alone, we implemented the routing agent using **Uniform Cost Search (UCS)**

**Execution Steps:**
1. Initialize a priority queue (frontier) with the starting node (Nairobi).

2. Expand nodes by checking adjacent towns.

3. Add the exact distance from Google Maps to the total path cost.

4. Continue expanding the lowest-cost path until the Kisumu node is reached and extracted from the frontier.

5. Compare standard routes mathematically using the accumulated distance to prove optimality.
   
## 4. Group Work Evidence
### Meeting Minutes
- **Date**: 07/10/2026
- **Platform**: Google Meet
- **Members Present**: Albert Miricho (189990), Sean Ndung'u (191602), Sydney Aisha (190497)
- **Agenda**:
  1. Problem formulation and PEAS definition.
  2. Construction of a highway-directed graph and abstraction tree.
  3. Distance verification on Google Maps.
  4. Repository fork verification and Markdown documentation compilation.

#### Team Photo
(./team_photo.jpg)


In [1]:
import heapq

def run_uniform_cost_search():
    # Adjacency list representation of the directed road network graph
    # Costs represent driving distances in kilometers extracted from Google Maps
    graph = {
        'Nairobi': [('Gitaru', 22.3), ('Ruaka', 15.5), ('Kiambu', 15.5), ('Ruiru', 24.6), ('Maai-Mahiu', 53.8)],
        'Gitaru': [('Limuru', 20.1)],
        'Ruaka': [('Limuru', 21.0)],
        'Kiambu': [('Limuru', 25.7)],
        'Ruiru': [('Thika', 21.4)],
        'Limuru': [('Naivasha', 56.1)],
        'Naivasha': [('Gilgil', 30.4)],
        'Gilgil': [('Nakuru', 42.6)],
        'Nakuru': [('Timboroa', 94.5), ('Eldama-Ravine', 63.1), ('Kericho', 107.0)],
        'Timboroa': [('Eldoret', 61.7)],
        'Eldama-Ravine': [('Eldoret', 108.0)],
        'Eldoret': [('Kapsabet', 46.0), ('Kakamega', 106.0)],
        'Kapsabet': [('Kisumu', 80.3)],
        'Kakamega': [('Kisumu', 53.4)],
        'Thika': [('Sagana', 63.9), ('Othaya', 86.8)],
        'Sagana': [('Nyahururu', 156.0)],
        'Othaya': [('Nyeri', 16.9)],
        'Nyeri': [('Nyahururu', 104.0)],
        'Nyahururu': [('Nakuru', 63.3)],
        'Kericho': [('Ahero', 60.9)],
        'Maai-Mahiu': [('Narok', 89.1)],
        'Narok': [('Bomet', 81.0)],
        'Bomet': [('Kisii', 82.4)],
        'Kisii': [('Homa-Bay', 52.0), ('Awach', 77.4)],
        'Homa-Bay': [('Ahero', 84.5)],
        'Awach': [('Ahero', 14.1)],
        'Ahero': [('Kisumu', 23.5)],
        'Kisumu': []
    }

    start_node = 'Nairobi'
    goal_node = 'Kisumu'

    # Priority Queue elements: (cumulative_cost, current_node, path_taken)
    frontier = []
    heapq.heappush(frontier, (0.0, start_node, [start_node]))

    # Dictionary to store the minimum cost to reach each explored node
    explored = {}

    print(f"UNIFORM COST SEARCH TRAVERSAL: {start_node} -> {goal_node}")

    step = 1
    while frontier:
        # Step 1: Pop the path with the smallest cumulative cost g(n)
        current_cost, current_node, path = heapq.heappop(frontier)

        # Print state log
        print(f"\nStep {step}: Expanding Node '{current_node}'")
        print(f"Current Path Cost g(n): {current_cost:.1f} km")
        print(f"Path Taken: {' -> '.join(path)}")

        # Goal Check: Goal test is performed when node is selected for expansion
        if current_node == goal_node:
            print("GOAL REACHED!")
            print(f"Optimal Path: {' -> '.join(path)}")
            print(f"Total Path Cost: {current_cost:.1f} km")
            return path, current_cost

        # Mark node as visited with its minimal cost
        if current_node in explored and explored[current_node] <= current_cost:
            step += 1
            continue
        explored[current_node] = current_cost

        # Expand neighbors
        for neighbor, edge_cost in graph.get(current_node, []):
            new_cost = current_cost + edge_cost
            if neighbor not in explored or new_cost < explored[neighbor]:
                heapq.heappush(frontier, (new_cost, neighbor, path + [neighbor]))
                print(f" -> Added to Frontier: {neighbor} (g = {new_cost:.1f} km)")

        step += 1

    print("Goal not reachable.")
    return None, float('inf')

if __name__ == '__main__':
    run_uniform_cost_search()

UNIFORM COST SEARCH TRAVERSAL: Nairobi -> Kisumu

Step 1: Expanding Node 'Nairobi'
Current Path Cost g(n): 0.0 km
Path Taken: Nairobi
 -> Added to Frontier: Gitaru (g = 22.3 km)
 -> Added to Frontier: Ruaka (g = 15.5 km)
 -> Added to Frontier: Kiambu (g = 15.5 km)
 -> Added to Frontier: Ruiru (g = 24.6 km)
 -> Added to Frontier: Maai-Mahiu (g = 53.8 km)

Step 2: Expanding Node 'Kiambu'
Current Path Cost g(n): 15.5 km
Path Taken: Nairobi -> Kiambu
 -> Added to Frontier: Limuru (g = 41.2 km)

Step 3: Expanding Node 'Ruaka'
Current Path Cost g(n): 15.5 km
Path Taken: Nairobi -> Ruaka
 -> Added to Frontier: Limuru (g = 36.5 km)

Step 4: Expanding Node 'Gitaru'
Current Path Cost g(n): 22.3 km
Path Taken: Nairobi -> Gitaru
 -> Added to Frontier: Limuru (g = 42.4 km)

Step 5: Expanding Node 'Ruiru'
Current Path Cost g(n): 24.6 km
Path Taken: Nairobi -> Ruiru
 -> Added to Frontier: Thika (g = 46.0 km)

Step 6: Expanding Node 'Limuru'
Current Path Cost g(n): 36.5 km
Path Taken: Nairobi -> Ruaka